# CS115 Team 05 - Policy Gradient for Reinforcement Learning

**Topic:** REINFORCE on `CartPole-v1`  
**Course:** CS115 - Mathematics for Computer Science  
**Team 05:** Đặng Chí Thanh, Hoàng Cao Sơn, Nguyễn Đức Ý, Phạm Vũ Xuân Quỳnh

This notebook is a runnable companion for the source-code submission. It follows the same flow as the project report:

1. Problem and objective.
2. Mathematical foundation.
3. REINFORCE implementation mapping.
4. Final checkpoint evaluation.
5. Experiment/result visualization.

The final report and presentation slides are submitted outside this source-code package. This notebook focuses on reproducibility and demonstration.


# Problem

The goal is to train a stochastic policy for `CartPole-v1`. At each time step, the agent observes a 4-dimensional state and chooses one of two actions:

- `0`: push cart left
- `1`: push cart right

The environment gives reward `+1` for every time step the pole remains balanced. The learning objective is to maximize expected return:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]
$$

where $\theta$ are the parameters of the policy network.


# Initial setup

The notebook reuses the project modules under `sources/`. It does not duplicate the algorithm implementation.


In [ ]:
from pathlib import Path

repo = Path("/content/CS115-Team05-PG")

if not repo.exists():
    !git clone https://github.com/uit-25730067-chithanh/CS115-Team05-PG.git /content/CS115-Team05-PG

%cd /content/CS115-Team05-PG
!pip install -q -r requirements.txt

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
SOURCE_DIR = PROJECT_ROOT / "sources"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

# Keep Matplotlib cache inside the project tmp folder when possible.
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "tmp" / "mpl-cache"))

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

from reinforce import compute_returns
from models.policy import PolicyNetwork

print("Project root:", PROJECT_ROOT)
print("Source dir exists:", SOURCE_DIR.exists())
print("reinforce.py exists:", (SOURCE_DIR / "reinforce.py").exists())
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)


# Environment sanity check

`CartPole-v1` has:

- state space $s_t \in \mathbb{R}^4$
- action space $a_t \in \{0, 1\}$
- maximum episode length 500 steps


In [ ]:
env = gym.make("CartPole-v1")
state, info = env.reset(seed=123)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial state shape:", state.shape)
print("Initial state:", np.round(state, 4))

env.close()


# Mathematical foundation

REINFORCE is based on the Policy Gradient Theorem:

$$
\nabla_\theta J(\theta)
= \mathbb{E}_{\tau \sim \pi_\theta}
\left[
\sum_{t=0}^{T}\nabla_\theta \log \pi_\theta(a_t \mid s_t)G_t
\right]
$$

The reward-to-go is:

$$
G_t = \sum_{k=t}^{T}\gamma^{k-t}r_{k+1}
$$

Since PyTorch optimizers minimize loss, the implementation uses:

$$
L(\theta) = -\sum_{t=0}^{T}\log \pi_\theta(a_t \mid s_t)G_t
$$

Minimizing $L(\theta)$ is equivalent to maximizing $J(\theta)$.


# Code mapping: reward-to-go

The project implementation computes returns in `sources/reinforce.py`. This quick check maps the formula to a simple episode with rewards `[1, 1, 1]` and $\gamma=0.99$.


In [ ]:
returns = compute_returns([1.0, 1.0, 1.0], gamma=0.99)
expected = [2.9701, 1.99, 1.0]

print("Computed returns:", returns)
print("Expected returns:", expected)

assert np.allclose(returns, expected, atol=1e-6)


# Code mapping: stochastic policy network

`PolicyNetwork` is a small MLP:

1. Linear layer from state dimension to hidden dimension.
2. ReLU activation.
3. Linear layer to action logits.
4. Softmax to produce action probabilities.

During training, actions are sampled from the probability distribution. During evaluation, the policy can use greedy `argmax` actions.


In [ ]:
torch.manual_seed(123)
policy = PolicyNetwork(state_dim=4, action_dim=2, hidden_dim=8)
sample_state = torch.zeros((1, 4), dtype=torch.float32)
probabilities = policy(sample_state)

print("Action probabilities:", probabilities.detach().numpy().round(4))
print("Probability sum:", float(probabilities.sum()))

assert probabilities.shape == (1, 2)
assert torch.all(probabilities >= 0)
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(1), atol=1e-6)


# Final artifacts

The public source package keeps one curated final run under `outputs/final/`:

- trained checkpoints
- reward logs
- training curve
- random baseline logs
- evaluation logs

These artifacts let the notebook demonstrate results without retraining for 1000 episodes.


In [ ]:
FINAL_DIR = PROJECT_ROOT / "outputs" / "final"
RUN_DIR = FINAL_DIR / "reinforce-cartpole-v1"
BASELINE_DIR = FINAL_DIR / "random-baseline"
EVAL_DIR = FINAL_DIR / "evaluation"

required_paths = [
    RUN_DIR / "best_policy.pth",
    RUN_DIR / "final_policy.pth",
    RUN_DIR / "rewards.txt",
    RUN_DIR / "metrics.txt",
    RUN_DIR / "run_config.txt",
    RUN_DIR / "training_curve.png",
    BASELINE_DIR / "baseline_stats.txt",
    BASELINE_DIR / "baseline_log.txt",
    BASELINE_DIR / "baseline_curve.png",
    EVAL_DIR / "eval_stats.txt",
    EVAL_DIR / "eval_log.txt",
]

missing = [path for path in required_paths if not path.exists()]
assert not missing, f"Missing final artifacts: {missing}"

print("Final artifacts ready:", len(required_paths), "files")


In [ ]:
def read_key_value_file(path):
    data = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        data[key.strip()] = value.strip()
    return data

run_config = read_key_value_file(RUN_DIR / "run_config.txt")
metrics = read_key_value_file(RUN_DIR / "metrics.txt")
eval_stats = read_key_value_file(EVAL_DIR / "eval_stats.txt")
baseline_stats = read_key_value_file(BASELINE_DIR / "baseline_stats.txt")

print("Run config:", run_config)
print("Training metrics:", metrics)
print("Evaluation stats:", eval_stats)
print("Random baseline stats:", baseline_stats)


# Training curve

The raw reward curve can be noisy because REINFORCE is a Monte-Carlo policy-gradient method. A moving average makes the convergence trend easier to read.


In [ ]:
def load_rewards(path):
    rewards = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split()
        rewards.append(float(parts[-1]))
    return np.array(rewards, dtype=np.float32)

rewards = load_rewards(RUN_DIR / "rewards.txt")
window = 50
moving_average = np.convolve(rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(rewards, alpha=0.25, label="Raw reward")
plt.plot(np.arange(window - 1, len(rewards)), moving_average, label="Moving average (50)")
plt.axhline(500, linestyle="--", linewidth=1, color="gray", label="CartPole-v1 max reward")
plt.title("REINFORCE Training Curve on CartPole-v1")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Episodes:", len(rewards))
print("Mean last 50:", float(np.mean(rewards[-50:])))
print("Best reward:", float(np.max(rewards)))


# Baseline vs trained policy

The random baseline provides context. The trained policy should reach much higher average return than random action selection.


In [ ]:
baseline_mean = float(baseline_stats["Mean"])
trained_eval_mean = float(eval_stats["mean_reward"])

plt.figure(figsize=(6, 4))
plt.bar(["Random baseline", "Trained policy"], [baseline_mean, trained_eval_mean], color=["#9ca3af", "#2563eb"])
plt.ylabel("Mean reward")
plt.title("Baseline vs Trained Policy")
plt.ylim(0, 520)
for idx, value in enumerate([baseline_mean, trained_eval_mean]):
    plt.text(idx, value + 10, f"{value:.1f}", ha="center")
plt.grid(axis="y", alpha=0.25)
plt.show()

print(f"Random baseline mean: {baseline_mean:.2f}")
print(f"Trained policy eval mean: {trained_eval_mean:.2f}")
assert trained_eval_mean > baseline_mean


# Load and evaluate the final checkpoint

This cell loads `best_policy.pth` and runs a short deterministic evaluation. It is intentionally short so the notebook remains fast during grading.


In [ ]:
def select_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def evaluate_checkpoint(checkpoint_path, episodes=3, seed=123):
    device = select_device()
    env = gym.make("CartPole-v1")
    if hasattr(env.action_space, "seed"):
        env.action_space.seed(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    hidden_dim = int(run_config["hidden_dim"])

    model = PolicyNetwork(state_dim, action_dim, hidden_dim=hidden_dim).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()

    episode_rewards = []
    with torch.no_grad():
        for episode in range(episodes):
            state, _ = env.reset(seed=seed if episode == 0 else None)
            done = False
            truncated = False
            total_reward = 0.0

            while not (done or truncated):
                state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(device)
                probs = model(state_tensor)
                action = torch.argmax(probs, dim=-1).item()
                state, reward, done, truncated, _ = env.step(action)
                total_reward += reward

            episode_rewards.append(total_reward)

    env.close()
    return np.array(episode_rewards, dtype=np.float32)

short_eval_rewards = evaluate_checkpoint(RUN_DIR / "best_policy.pth", episodes=3, seed=123)
print("Short evaluation rewards:", short_eval_rewards.tolist())
print("Short evaluation mean:", float(short_eval_rewards.mean()))
assert short_eval_rewards.mean() > baseline_mean


# Optional: retraining command

The notebook does not retrain by default because a full 1000-episode run can take time on a grader's machine. To reproduce the final run from the terminal:

```bash
python3 scripts/train.py --episodes 1000 --seed 123 --hidden-dim 128
```

To evaluate the saved final checkpoint:

```bash
python3 scripts/evaluate.py --checkpoint outputs/final/reinforce-cartpole-v1/best_policy.pth --episodes 10 --hidden-dim 128
```


# Conclusion

This notebook demonstrates the full project path:

- The mathematical objective is expected return maximization.
- REINFORCE uses the log-derivative trick and Monte-Carlo reward-to-go.
- The source implementation maps directly to the policy-gradient formula.
- The trained `CartPole-v1` policy substantially outperforms the random baseline.
- Final results are reproducible through project scripts and curated artifacts under `outputs/final/`.

Main limitations:

- REINFORCE has high variance because it uses Monte-Carlo episode returns.
- Performance depends on seed, learning rate, hidden dimension, and number of episodes.
- The notebook is a runnable companion; detailed proofs and formal discussion are in the final report submitted separately.


In [ ]:
coverage_check = {
    "problem": True,
    "math_foundation": True,
    "code_mapping": True,
    "environment": True,
    "final_artifacts": True,
    "training_curve": True,
    "baseline_comparison": trained_eval_mean > baseline_mean,
    "checkpoint_evaluation": short_eval_rewards.mean() > baseline_mean,
}

for item, ok in coverage_check.items():
    print(f"{item}: {'OK' if ok else 'MISSING'}")

assert all(coverage_check.values())
